# Demo 7 — Model Recovery / Model Stealing
## KnockoffNets: Cloning a Black-Box Classifier

**Threat Model:** Black-box query-based — attacker sees only **hard labels**
(the most restrictive API setting)

**Goal:** Train a **functionally equivalent clone** of a victim model,
violating the owner's **IP and monetisation** (e.g., MLaaS APIs).

**Based on:** Knockoff Nets (Orekondy et al., CVPR 2019)

In [ ]:
# Install
# !pip install torch torchvision matplotlib numpy

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
torch.manual_seed(42)
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([transforms.ToTensor()])
full_train_f = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set_f   = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
full_train = torch.utils.data.Subset(full_train_f, range(1000))
test_set   = torch.utils.data.Subset(test_set_f, range(200))
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False)

## Step 1 — Train the Victim (Proprietary) Model

In [ ]:
class VictimModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*7*7,256), nn.ReLU(), nn.Linear(256,10)
        )
    def forward(self, x): return self.net(x)

victim = VictimModel().to(DEVICE)
train_loader = torch.utils.data.DataLoader(full_train, batch_size=128, shuffle=True)
opt = optim.Adam(victim.parameters(), lr=1e-3); fn = nn.CrossEntropyLoss()
print('Training victim model (kept secret from attacker)...')
for epoch in range(3):
    victim.train()
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); fn(victim(X), y).backward(); opt.step()
victim.eval()
correct = sum((victim(X.to(DEVICE)).argmax(1)==y.to(DEVICE)).sum().item()
               for X, y in test_loader)
victim_acc = correct / len(test_set)
print(f'Victim accuracy: {victim_acc*100:.2f}%')

## Step 2 — Model Stealing with Different Query Budgets

We sweep different **query budgets** to show how clone fidelity grows
as the attacker makes more API calls.

In [ ]:
class CloneModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(16*14*14,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self, x): return self.net(x)

def steal_model(query_budget):
    indices = np.random.choice(len(full_train), query_budget, replace=False)
    query_imgs = torch.stack([full_train[i][0] for i in indices])
    victim.eval()
    with torch.no_grad():
        stolen_labels = victim(query_imgs.to(DEVICE)).argmax(1).cpu()
    clone_ds     = torch.utils.data.TensorDataset(query_imgs, stolen_labels)
    clone_loader = torch.utils.data.DataLoader(clone_ds, batch_size=64, shuffle=True)
    clone    = CloneModel().to(DEVICE)
    opt_c    = optim.Adam(clone.parameters(), lr=1e-3)
    for _ in range(10):
        clone.train()
        for Xb, yb in clone_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt_c.zero_grad(); fn(clone(Xb), yb).backward(); opt_c.step()
    clone.eval()
    agree = 0
    with torch.no_grad():
        for X, _ in test_loader:
            X = X.to(DEVICE)
            agree += (clone(X).argmax(1) == victim(X).argmax(1)).sum().item()
    fidelity = agree / len(test_set)
    acc = sum((clone(X.to(DEVICE)).argmax(1)==y.to(DEVICE)).sum().item()
               for X, y in test_loader) / len(test_set)
    return fidelity, acc

QUERY_BUDGETS  = [200, 500, 1000, 3000, 5000]
clone_results  = {}
print(f'{'Budget':>8}  {'Clone Acc':>10}  {'Fidelity':>10}')
print('-' * 35)
for budget in QUERY_BUDGETS:
    fidelity, acc = steal_model(budget)
    clone_results[budget] = (acc, fidelity)
    print(f'{budget:>8}  {acc*100:>9.2f}%  {fidelity*100:>9.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
budgets    = list(clone_results.keys())
accs       = [clone_results[b][0]*100 for b in budgets]
fidelities = [clone_results[b][1]*100 for b in budgets]

axes[0].plot(budgets, accs, 'o-', color='#e74c3c', linewidth=2, markersize=7, label='Clone Acc')
axes[0].axhline(victim_acc*100, linestyle='--', color='#27ae60', linewidth=1.5, label='Victim Acc')
axes[0].set_xlabel('Query Budget (# API calls)')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Clone Accuracy vs Query Budget', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(budgets, fidelities, 's-', color='#3498db', linewidth=2, markersize=7)
axes[1].set_xlabel('Query Budget (# API calls)')
axes[1].set_ylabel('Fidelity (% agreement with victim)')
axes[1].set_title('Clone Fidelity vs Query Budget', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.suptitle('Model Recovery / Stealing (KnockoffNets-style)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('07_model_stealing_results.png', dpi=150)
plt.show()

## Summary

Model stealing shows that **ML models can be replicated without training data**.
With ~5000 queries, a clone achieves high fidelity, stealing the victim's IP.

**Real-world impact:** Bypasses API paywalls, enables further white-box attacks.

**Defenses:** Query rate-limiting, output perturbation, prediction watermarking,
anomalous query pattern detection.

In [ ]:
# --- Visualizing Model Fidelity (Shared Mistakes) ---
# To truly show what the clone model learned, let's visualize its predictions side-by-side with the victim.
# A high-fidelity clone will not only get the same images right, but it will make the *exact same mistakes*.

print("Training a quick demo clone (budget=3000) for visualization...")
budget = 3000
indices = np.random.choice(len(full_train), budget, replace=False)
query_imgs = torch.stack([full_train[i][0] for i in indices])

victim.eval()
with torch.no_grad():
    stolen_labels = victim(query_imgs.to(DEVICE)).argmax(1).cpu()

clone_ds = torch.utils.data.TensorDataset(query_imgs, stolen_labels)
clone_loader = torch.utils.data.DataLoader(clone_ds, batch_size=64, shuffle=True)
demo_clone = CloneModel().to(DEVICE)
opt_c = optim.Adam(demo_clone.parameters(), lr=1e-3)
fn_loss = nn.CrossEntropyLoss()
for _ in range(10):
    demo_clone.train()
    for Xb, yb in clone_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        opt_c.zero_grad(); fn_loss(demo_clone(Xb), yb).backward(); opt_c.step()

demo_clone.eval()

# Find and plot interesting cases
shared_mistakes = []
regular_cases = []

# We will search the test set for images where the Victim makes a mistake,
# to see if the Clone makes the *exact same mistake*.
for i in range(len(test_set)):
    img, true_label = test_set[i]
    with torch.no_grad():
        v_pred = victim(img.unsqueeze(0).to(DEVICE)).argmax(1).item()
        c_pred = demo_clone(img.unsqueeze(0).to(DEVICE)).argmax(1).item()
        
    # Check if it is a shared mistake (victim got it wrong, clone copied the wrong answer)
    if v_pred != true_label and c_pred == v_pred:
        shared_mistakes.append((img, true_label, v_pred, c_pred))
    else:
        regular_cases.append((img, true_label, v_pred, c_pred))
        
    if len(shared_mistakes) >= 10:
        break

# Combine them to ensure we always plot exactly 10 images
cases_to_plot = shared_mistakes + regular_cases
cases_to_plot = cases_to_plot[:10]

fig, axes = plt.subplots(2, 5, figsize=(14, 7))
fig.suptitle('Victim vs Clone Predictions (Showing Fidelity & Shared Mistakes)', fontsize=15, fontweight='bold')

for idx, (img, true_label, v_pred, c_pred) in enumerate(cases_to_plot):
    ax = axes[idx//5, idx%5]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.axis('off')
    
    v_color = 'green' if v_pred == true_label else 'red'
    # Clone color is blue if it successfully mimicked the victim, orange if they disagree
    c_color = 'blue' if c_pred == v_pred else 'orange'
    
    ax.text(0.5, -0.1, f"Victim Pred: {v_pred}", color=v_color, transform=ax.transAxes, ha='center', fontweight='bold', fontsize=12)
    ax.text(0.5, -0.25, f"Clone Pred: {c_pred}", color=c_color, transform=ax.transAxes, ha='center', fontweight='bold', fontsize=12)
    ax.set_title(f"True Label: {true_label}", fontsize=12)

plt.tight_layout()
plt.subplots_adjust(hspace=0.5)
plt.savefig('07_fidelity_viz.png', dpi=150)
plt.show()